In [ ]:
import numpy as np
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 데이터 로드 함수
def load_data(system_types, V, max_samples=None):
    data = []
    labels = []
    for system_type in system_types:
        base_dir = f'./data_set/{system_type}/V{V}/'
        for parm_dir in os.listdir(base_dir):
            parm_path = os.path.join(parm_path, parm_dir)
            sample_files = os.listdir(parm_path)
            if max_samples is not None:
                sample_files = sample_files[:max_samples]  # 최대 샘플 수만큼 자르기
            
            for sample_file in sample_files:
                sample_data = np.load(os.path.join(parm_path, sample_file), allow_pickle=True).item()
                x_data = sample_data['samples'][:, 0]  # x 데이터만 사용
                data.append(x_data.flatten())
                labels.append(system_type)
    return np.array(data), np.array(labels)

# Option 1: Beta, Epsilon 상관없이 전체 데이터 나누기
def option1_split(data, labels):
    return train_test_split(data, labels, test_size=0.5, random_state=42)

# Option 2: 각 Beta, Epsilon 쌍을 나눠서 train/test split
def option2_split(data, labels, system_types, V, max_samples=None):
    train_data, test_data = [], []
    train_labels, test_labels = [], []
    for system_type in system_types:
        base_dir = f'./data_set/{system_type}/V{V}/'
        for parm_dir in os.listdir(base_dir):
            parm_path = os.path.join(base_dir, parm_dir)
            sample_files = os.listdir(parm_path)
            if max_samples is not None:
                sample_files = sample_files[:max_samples]  # 최대 샘플 수만큼 자르기
            
            samples = []
            for sample_file in sample_files:
                sample_data = np.load(os.path.join(parm_path, sample_file), allow_pickle=True).item()
                x_data = sample_data['samples'][:, 0]  # x 데이터만 사용
                samples.append(x_data.flatten())
            samples = np.array(samples)
            half = len(samples) // 2
            train_data.extend(samples[:half])
            test_data.extend(samples[half:])
            train_labels.extend([system_type] * half)
            test_labels.extend([system_type] * (len(samples) - half))
    return np.array(train_data), np.array(test_data), np.array(train_labels), np.array(test_labels)

# Label encoding
def encode_labels(labels):
    label_dict = {label: idx for idx, label in enumerate(np.unique(labels))}
    encoded_labels = np.array([label_dict[label] for label in labels])
    return encoded_labels, label_dict

# Large Kernel CNN 모델 정의
def build_large_kernel_cnn(input_shape, num_classes):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv1D(filters=64, kernel_size=25, strides=1, padding='same', activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(filters=128, kernel_size=25, strides=1, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# CNN 학습 및 평가 함수
def evaluate_cnn(train_data, train_labels, test_data, test_labels, input_shape, num_classes):
    model = build_large_kernel_cnn(input_shape, num_classes)
    model.fit(train_data, train_labels, epochs=10, batch_size=32, validation_split=0.2, verbose=2)
    predictions = model.predict(test_data)
    predicted_labels = np.argmax(predictions, axis=1)
    cm = confusion_matrix(test_labels, predicted_labels)
    return cm, model, predicted_labels

# Confusion matrix 시각화 함수
def plot_confusion_matrix(cm, title, label_dict):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(label_dict.keys()))
    disp.plot(cmap=plt.cm.Blues)
    plt.title(title)
    plt.show()

# 잘못 예측한 샘플 시각화 함수
def visualize_misclassified(test_data, test_labels, predictions, n_samples=9):
    misclassified_indices = np.where(test_labels != predictions)[0]
    
    if len(misclassified_indices) == 0:
        print("No misclassified samples to visualize.")
        return
    
    n_samples = min(len(misclassified_indices), n_samples)
    plt.figure(figsize=(10, 10))
    
    for i, idx in enumerate(misclassified_indices[:n_samples]):
        plt.subplot(3, 3, i + 1)
        plt.plot(test_data[idx])
        plt.title(f"True: {test_labels[idx]}, Pred: {predictions[idx]}")
    
    plt.tight_layout()
    plt.show()

# 메인 실행부
if __name__ == "__main__":
    system_types = ['B', 'SSS', 'OSC']
    V = 10**8
    max_samples = 20  # 가져올 최대 샘플 수 설정
    
    # 데이터 로드
    data, labels = load_data(system_types, V, max_samples)
    
    # 레이블 인코딩
    encoded_labels, label_dict = encode_labels(labels)
    
    # 데이터 차원 추가 (CNN 입력을 위해)
    data = np.expand_dims(data, axis=-1)
    input_shape = (data.shape[1], 1)
    
    # Option 1: Beta, Epsilon 상관없이 전체 데이터 나누기
    train_data, test_data, train_labels, test_labels = option1_split(data, encoded_labels)
    cm1, model1, predictions1 = evaluate_cnn(train_data, train_labels, test_data, test_labels, input_shape, len(label_dict))
    plot_confusion_matrix(cm1, 'Confusion Matrix - Option 1 (Overall Data Split)', label_dict)
    visualize_misclassified(test_data, test_labels, predictions1)
    
    # Option 2: 각 Beta, Epsilon 쌍을 나눠서 train/test split
    train_data, test_data, train_labels, test_labels = option2_split(data, labels, system_types, V, max_samples)
    train_labels, _ = encode_labels(train_labels)
    test_labels, _ = encode_labels(test_labels)
    cm2, model2, predictions2 = evaluate_cnn(train_data, train_labels, test_data, test_labels, input_shape, len(label_dict))
    plot_confusion_matrix(cm2, 'Confusion Matrix - Option 2 (Beta-Epsilon Pair Split)', label_dict)
    visualize_misclassified(test_data, test_labels, predictions2)


UnboundLocalError: cannot access local variable 'parm_path' where it is not associated with a value